<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🔀 Lab 03b: Build a Multi-Agent Travel Workflow</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 What We Learn

In this lab, we orchestrate multiple specialist agents into a **multi-agent workflow** using `WorkflowAgentDefinition`. We create three domain-specific agents — **Flight Agent**, **Hotel Agent**, and **Car Rental Agent** — and wire them together with a YAML-based workflow that routes customer queries through the appropriate specialists.

> **Think of this as a team of specialists.** Instead of one generalist who knows a little about everything, you assemble focused experts — a flight advisor, a hotel concierge, and a car rental desk — each handling their domain. The workflow is the dispatcher that routes each customer request to the right specialist and combines their answers.

---


## 🧳 The Contoso Travel Story

The Contoso Travel agent from Lab 03a works — it can search real flights, hotels, and car rentals. But as the team added more features, problems started emerging. The system prompt grew to over 2,000 words trying to cover flight booking rules, hotel amenity details, car rental policies, and general travel advice all at once. During testing, a customer asked: *"Plan a trip from NYC to London — I need a flight, a nice hotel near the theater district, and a rental car for day trips."* The agent searched for flights... then seemed to forget about the hotel request. When reminded, it searched hotels but gave car rental advice mixed into the hotel description. **The single agent was trying to be an expert in everything and failing at the details.**

### 🔴 The Problem You Face Right Now

- **One agent can't do everything well.** As you add more domains (flights, hotels, cars) and more complex logic, a single agent's instructions become bloated and contradictory.
- It loses focus, confuses tool calls, and produces inconsistent results.
- Real travel agencies have specialist advisors for a reason — you need the same architecture in your AI system.
- But how do you break one agent into multiple specialists and coordinate them?

### ✅ What This Lab Solves

This lab introduces **Multi-Agent Workflows** using `WorkflowAgentDefinition`:
 - creating three specialist agents — a Flight Agent, a Hotel Agent, and a Car Rental Agent — each with focused instructions and domain-specific tools,
 - wiring them together with a YAML-based workflow, and
 - routing customer queries to the right specialist and aggregating their responses into a unified trip plan.

## 2. Setup

---


In [ ]:
# Setup: imports and Microsoft Foundry client config
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    # Orchestrates multiple agents via a YAML workflow
    WorkflowAgentDefinition,
    FunctionTool,
    Tool,
)

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
# allow_preview=True is REQUIRED for WorkflowAgentDefinition
project_client = AIProjectClient(endpoint=endpoint, credential=credential, allow_preview=True)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Microsoft Foundry (preview features enabled)")

## 3. Create Specialist Agents

We create three focused agents — each an expert in one travel domain — plus a **concierge agent** that combines their results into a unified trip plan.

```
Customer Query
  ├─→ ✈️ Flight Agent     → flight options
  ├─→ 🏨 Hotel Agent      → hotel options
  ├─→ 🚗 Car Agent        → car rental options
  └─→ 🎩 Concierge Agent  → combines all into final trip plan
```

---


In [ ]:
# Create 4 PromptAgentDefinitions.
# 3 specialist agents + 1 concierge that combines their results.
# create_version() registers the agent server-side and
# returns an object with .name and .version for referencing.

# Flight Specialist Agent
flight_agent = project_client.agents.create_version(
    agent_name="contoso-flight-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""You are the Contoso Travel Flight Specialist. 
You help customers find flights. When given a travel request:
1. Identify the origin and destination cities
2. Suggest flight options with approximate pricing
3. Ask about cabin class preference if not specified
4. Be concise and format your response clearly with bullet points.
Always end your response with: [FLIGHT SEARCH COMPLETE]""",
    ),
)
print(f"✅ Flight Agent created: {flight_agent.name} (v{flight_agent.version})")

# Hotel Specialist Agent
hotel_agent = project_client.agents.create_version(
    agent_name="contoso-hotel-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""You are the Contoso Travel Hotel Specialist.
You help customers find hotels. When given a travel request:
1. Identify the destination city
2. Suggest hotel options across different star ratings and price points
3. Highlight key amenities
4. Be concise and format your response clearly with bullet points.
Always end your response with: [HOTEL SEARCH COMPLETE]""",
    ),
)
print(f"✅ Hotel Agent created: {hotel_agent.name} (v{hotel_agent.version})")

# Car Rental Specialist Agent
car_agent = project_client.agents.create_version(
    agent_name="contoso-car-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""You are the Contoso Travel Car Rental Specialist.
You help customers find rental cars. When given a travel request:
1. Identify the city where a car is needed
2. Suggest car types with pricing
3. Mention availability and pickup details
4. Be concise and format your response clearly with bullet points.
Always end your response with: [CAR RENTAL SEARCH COMPLETE]""",
    ),
)
print(f"✅ Car Rental Agent created: {car_agent.name} (v{car_agent.version})")

# Concierge Agent — combines all specialist responses
concierge_agent = project_client.agents.create_version(
    agent_name="contoso-concierge-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""You are the Contoso Travel Concierge. You receive 
research from three specialist agents (flights, hotels, car rentals) and
combine their findings into a single, well-organized trip plan for the customer.

Format your response as:
## ✈️ Flights
(summarize the flight options)

## 🏨 Hotels
(summarize the hotel options)

## 🚗 Car Rentals
(summarize the car rental options)

## 📋 Trip Summary
(brief overview with total estimated cost range)

Be concise but comprehensive. Present the best options clearly.""",
    ),
)
print(f"✅ Concierge Agent created: {concierge_agent.name} (v{concierge_agent.version})")

## 4. Define the Workflow YAML

The workflow routes the customer's request through each specialist agent, then passes all three responses to the **concierge agent** who combines them into a unified trip plan.

---


In [ ]:
# Define the YAML-based workflow that orchestrates agents.
# The workflow creates isolated conversations for each specialist,
# invokes them in sequence, and stores their responses in variables.
#
# NOTE: SendActivity treats `activity` as literal text — it does NOT
# evaluate expressions. To get the actual agent response text, we
# retrieve the sub-conversation messages client-side after the
# workflow completes.

workflow_yaml = f"""
kind: workflow
trigger:
  kind: OnConversationStart
  id: contoso_travel_workflow
  actions:
    - kind: SetVariable
      id: capture_user_request
      variable: Local.LatestMessage
      value: "=UserMessage(System.LastMessageText)"

    - kind: CreateConversation
      id: create_flight_conversation
      conversationId: Local.FlightConversationId

    - kind: CreateConversation
      id: create_hotel_conversation
      conversationId: Local.HotelConversationId

    - kind: CreateConversation
      id: create_car_conversation
      conversationId: Local.CarConversationId

    - kind: InvokeAzureAgent
      id: invoke_flight_agent
      description: Search for flights
      conversationId: "=Local.FlightConversationId"
      agent:
        name: {flight_agent.name}
      input:
        messages: "=Local.LatestMessage"
      output:
        messages: Local.FlightResponse

    - kind: InvokeAzureAgent
      id: invoke_hotel_agent
      description: Search for hotels
      conversationId: "=Local.HotelConversationId"
      agent:
        name: {hotel_agent.name}
      input:
        messages: "=Local.LatestMessage"
      output:
        messages: Local.HotelResponse

    - kind: InvokeAzureAgent
      id: invoke_car_agent
      description: Search for car rentals
      conversationId: "=Local.CarConversationId"
      agent:
        name: {car_agent.name}
      input:
        messages: "=Local.LatestMessage"
      output:
        messages: Local.CarResponse

    - kind: EndConversation
      id: end_workflow
"""

print("✅ Workflow YAML defined")
print(f"   Specialists: {flight_agent.name}, {hotel_agent.name}, {car_agent.name}")
print(f"   Concierge: {concierge_agent.name} (called client-side after workflow)")

## 5. Create the Workflow Agent

Register the workflow with Microsoft Foundry.

---


In [ ]:
# WorkflowAgentDefinition takes YAML instead of instructions.
# Unlike PromptAgentDefinition (single LLM + prompt),
# a workflow agent orchestrates multiple agents via actions.
workflow = project_client.agents.create_version(
    agent_name="contoso-travel-workflow",
    definition=WorkflowAgentDefinition(workflow=workflow_yaml),
)

print(f"✅ Workflow Agent created!")
print(f"   Name: {workflow.name}")
print(f"   Version: {workflow.version}")
print(f"   ID: {workflow.id}")

## 6. Test: End-to-End Trip Planning

Send a trip planning request and observe the workflow orchestrating all three specialist agents.

---


In [ ]:
# Run the workflow via the OpenAI-compatible Responses API
conversation = openai_client.conversations.create()
print(f"✅ Conversation created (ID: {conversation.id})")

user_input = "Plan a trip from New York to London in September. I need flights, a hotel, and a car rental."

print("\n┌" + "─" * 70 + "┐")
print("│ 📨 INITIAL QUERY" + " " * 53 + "│")
print("├" + "─" * 70 + "┤")
for line in [user_input[i:i+68] for i in range(0, len(user_input), 68)]:
    print(f"│ {line:<68} │")
print("└" + "─" * 70 + "┘")

stream = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": workflow.name, "type": "agent_reference"}},
    input=user_input,
    stream=True,
)

response_id = None
actions_log = []

print("\n⏳ Workflow executing...\n")
for event in stream:
    if event.type == "response.output_item.added" and event.item.type == "workflow_action":
        print(f"  🔄 '{event.item.action_id}' started")
    elif event.type == "response.output_item.done" and event.item.type == "workflow_action":
        print(f"  ✅ '{event.item.action_id}' completed")
        actions_log.append(event.item.action_id)
    elif event.type == "response.completed":
        response_id = event.response.id

print(f"\n✅ Workflow completed! ({len(actions_log)} actions)")
print(f"   Response ID: {response_id}")

## 7. Retrieve & Combine Agent Responses

After the workflow orchestrates the specialist agents, we retrieve their actual responses from the conversation and pass them to a **concierge agent** that combines everything into a unified trip plan.

---


In [ ]:
# After the workflow, retrieve the actual specialist responses.
# The workflow stored them in sub-conversations — we can access
# them via the conversation items API.
response = openai_client.responses.retrieve(response_id=response_id)

# Collect sub-conversation IDs from workflow_action items
sub_conv_ids = []
for item in response.output:
    if item.type == "workflow_action":
        raw = item.model_dump()
        # CreateConversation actions may store the conversation ID
        for key, val in raw.items():
            if isinstance(val, str) and val.startswith("conv_"):
                sub_conv_ids.append((item.action_id, val))

# Try listing items from the main conversation
print("┌" + "─" * 70 + "┐")
print("│ 🔍 RETRIEVING AGENT RESPONSES" + " " * 40 + "│")
print("├" + "─" * 70 + "┤")

specialist_texts = {}
agent_labels = {"flight": "✈️  Flight", "hotel": "🏨 Hotel", "car": "🚗 Car Rental"}

try:
    items = openai_client.conversations.items.list(conversation_id=conversation.id)
    for ci in items.data:
        raw = ci.model_dump()
        role = raw.get("role", "")
        if role == "assistant":
            content = raw.get("content", [])
            if content and isinstance(content, list):
                for c in content:
                    text = c.get("text", "")
                    if text and len(text) > 20:
                        # Identify which agent by checking markers
                        for key, label in agent_labels.items():
                            if key.upper() in text.upper():
                                specialist_texts[key] = (label, text)
                                break
                        else:
                            specialist_texts[f"other_{len(specialist_texts)}"] = ("🤖 Agent", text)
except Exception:
    pass

# If conversation items didn't yield results, try sub-conversations
if not specialist_texts and sub_conv_ids:
    for action_id, conv_id in sub_conv_ids:
        try:
            items = openai_client.conversations.items.list(conversation_id=conv_id)
            for ci in items.data:
                raw = ci.model_dump()
                if raw.get("role") == "assistant":
                    content = raw.get("content", [])
                    if content and isinstance(content, list):
                        text = content[0].get("text", "")
                        if text:
                            specialist_texts[action_id] = (action_id, text)
        except Exception:
            pass

# Display what we found
if specialist_texts:
    for key in ["flight", "hotel", "car"]:
        if key in specialist_texts:
            label, text = specialist_texts[key]
            print(f"│ {label} Agent Response:" + " " * (54 - len(label)) + "│")
            print("│" + "─" * 70 + "│")
            for line in text.split('\n'):
                while len(line) > 68:
                    print(f"│ {line[:68]} │")
                    line = line[68:]
                print(f"│ {line:<68} │")
            print("│" + " " * 70 + "│")
    # Show any other texts we found
    for key, (label, text) in specialist_texts.items():
        if key not in ["flight", "hotel", "car"]:
            print(f"│ {label} Response:" + " " * (60 - len(label)) + "│")
            print("│" + "─" * 70 + "│")
            for line in text.split('\n'):
                while len(line) > 68:
                    print(f"│ {line[:68]} │")
                    line = line[68:]
                print(f"│ {line:<68} │")
            print("│" + " " * 70 + "│")
else:
    print("│ (Specialist responses stored in workflow-internal context)" + " " * 12 + "│")
    print("│ Calling concierge agent directly for combined trip plan..." + " " * 12 + "│")

print("└" + "─" * 70 + "┘")

# Call the concierge agent to produce a combined trip plan
print("\n🎩 Calling Concierge Agent for combined trip plan...\n")

concierge_input = user_input
if specialist_texts:
    parts = []
    for key in ["flight", "hotel", "car"]:
        if key in specialist_texts:
            label, text = specialist_texts[key]
            parts.append(f"--- {label} Agent ---\n{text}")
    concierge_input = (
        f"Customer request: {user_input}\n\n"
        "Here are the specialist agent responses:\n\n" +
        "\n\n".join(parts) +
        "\n\nPlease combine these into a unified trip plan."
    )

concierge_conv = openai_client.conversations.create()
concierge_stream = openai_client.responses.create(
    conversation=concierge_conv.id,
    extra_body={"agent_reference": {"name": concierge_agent.name, "type": "agent_reference"}},
    input=concierge_input,
    stream=True,
)

concierge_response_id = None
for event in concierge_stream:
    if event.type == "response.completed":
        concierge_response_id = event.response.id

concierge_response = openai_client.responses.retrieve(response_id=concierge_response_id)
final_text = ""
for item in concierge_response.output:
    if item.type == "message" and item.content:
        final_text = item.content[0].text
        break

print("┌" + "─" * 70 + "┐")
print("│ 🎩 COMBINED TRIP PLAN (Concierge Agent)" + " " * 30 + "│")
print("├" + "─" * 70 + "┤")
for line in final_text.split('\n'):
    while len(line) > 68:
        print(f"│ {line[:68]} │")
        line = line[68:]
    print(f"│ {line:<68} │")
print("└" + "─" * 70 + "┘")

# Clean up concierge conversation
openai_client.conversations.delete(conversation_id=concierge_conv.id)

## 8. Cleanup

---


In [ ]:
# Cleanup: delete remote resources in reverse order
openai_client.conversations.delete(conversation_id=conversation.id)
print("✅ Conversation deleted")

# Delete workflow first, then the agents it references
project_client.agents.delete_version(agent_name=workflow.name, agent_version=workflow.version)
print("✅ Workflow deleted")

project_client.agents.delete_version(agent_name=concierge_agent.name, agent_version=concierge_agent.version)
print("✅ Concierge Agent deleted")

project_client.agents.delete_version(agent_name=flight_agent.name, agent_version=flight_agent.version)
print("✅ Flight Agent deleted")

project_client.agents.delete_version(agent_name=hotel_agent.name, agent_version=hotel_agent.version)
print("✅ Hotel Agent deleted")

project_client.agents.delete_version(agent_name=car_agent.name, agent_version=car_agent.version)
print("✅ Car Rental Agent deleted")

openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed")

## 9. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Created three specialist agents (Flight, Hotel, Car Rental) each with domain-specific instructions</li>
  <li>Defined a YAML-based workflow that orchestrates all three agents</li>
  <li>Tested end-to-end trip planning with streaming workflow events</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>💡 Key Takeaway:</strong> Multi-agent workflows let you decompose complex tasks into specialist agents. Each agent focuses on what it does best, and the workflow handles orchestration. This is how production-grade agentic systems are built.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ Next:</strong> In <a href="lab-04-tracing.ipynb">Lab 04</a>, we'll add <b>OpenTelemetry tracing</b> to observe exactly what happens inside our travel agent — from the user query to tool calls to the final response!
</div>

---
